# Tracking Updates with UpdateTable

# Introduction

This tutorial demonstrates how to use ``BaseUpdateTableSchema`` and ``UpdateTableManifestation`` to create tables that track data changes using an incrementing ``update_id``. This is useful for synchronization protocols, event logs, or any scenario where you need to query "what changed since X".

## What is an UpdateTable?

An **UpdateTable** is a table that automatically includes an ``update_id`` column.
*   **Sequential IDs**: It provides mechanisms to track updates sequentially.
*   **Delta Queries**: It offers methods to query items *from* a specific update ID (inclusive or exclusive).
*   **Optimization**: Useful for efficient client synchronization (e.g., "give me everything new since update_id 50").

This tutorial will guide you through:
- Defining an Update Schema Mixin
- Creating a Manifestation
- Setting up the Database
- Inserting data with Update IDs
- Querying based on Update IDs

**Prerequisites:**
- Basic familiarity with Python and SQLAlchemy
- Installed package: ``sqlalchemyobjects``

## Table of Contents

- [Importing the Module](#Importing-the-Module)
- [Core Functionality](#Core-Functionality)
- [Usage](#Usage)
- [Async Usage](#Async-Usage)
- [API Highlights](#API-Highlights)
- [Troubleshooting / FAQs](#Troubleshooting-/-FAQs)
- [Conclusion and Next Steps](#Conclusion-and-Next-Steps)



# Importing the Module

We start by importing the necessary classes.



In [ ]:
from pathlib import Path
from sqlalchemy.orm import Mapped, DeclarativeBase
from sqlalchemy.ext.asyncio import AsyncAttrs

from sqlalchemyobjects import Database, BaseUpdateTableSchema, UpdateTableManifestation

# Core Functionality

## 1. Define the Schema Mixin

Inherit from ``BaseUpdateTableSchema``. This mixin automatically adds an ``update_id`` column (BigInteger).



In [ ]:
class EventLogSchema(BaseUpdateTableSchema):
    """Schema for an event log."""
    event_type: Mapped[str]
    payload: Mapped[str]

## 2. Define the Manifestation

The manifestation provides the query methods like ``get_from_update``.



In [ ]:
class EventLogManifestation(UpdateTableManifestation):
    """Manifestation class for Event Log table, supporting delta queries."""

## 3. Define the Database Schema

Combine the TableSchema with the Declarative Base.



In [ ]:
class DatabaseSchema(AsyncAttrs, DeclarativeBase):
    """Declarative base class for the database schema."""

class EventLogTable(EventLogSchema, DatabaseSchema):
    """SQLAlchemy table definition for Event Logs, with update tracking."""
    __tablename__ = "event_log"

## 4. Define the Database Class

Register the table in the ``table_map``.



In [ ]:
class LogDatabase(Database):
    """Database class managing the Event Log table."""
    schema = DatabaseSchema
    table_map = {
        "logs": (EventLogManifestation, EventLogTable, {})
    }

    @property
    def logs(self) -> EventLogManifestation:
        return self.tables["logs"]

## 5. Setup Database

Initialize the database.



In [ ]:
db_path = Path("tutorial_update.sqlite")
if db_path.exists():
    db_path.unlink()

database = LogDatabase(path=db_path)
database.create_database()

# Access the table
logs = database.logs

# Usage

## Inserting Data

When inserting, you must manage the ``update_id``. In many systems, this might come from a global counter or sequence. Here we manage it manually for demonstration.



In [ ]:
print("Inserting logs...")
# Simulating a sequence of events
logs.insert({"event_type": "LOGIN", "payload": "User A", "update_id": 1})
logs.insert({"event_type": "CLICK", "payload": "Button X", "update_id": 2})
logs.insert({"event_type": "LOGOUT", "payload": "User A", "update_id": 3})

## Querying Last Update ID

You can quickly check the highest ``update_id`` in the table.



In [ ]:
last_id = logs.get_last_update_id()
print(f"Last Update ID: {last_id}")

## Querying Deltas (From Update ID)

To get all changes since a specific point.



In [ ]:
# Get everything from ID 2 onwards (Inclusive)
print("Logs from ID 2 (inclusive):")
results = logs.get_from_update(update_id=2, inclusive=True)
for item in results.scalars():
    print(f"[{item.update_id}] {item.event_type}: {item.payload}")

# Get everything AFTER ID 2 (Exclusive)
print("\nLogs after ID 2 (exclusive):")
results = logs.get_from_update(update_id=2, inclusive=False)
for item in results.scalars():
    print(f"[{item.update_id}] {item.event_type}: {item.payload}")

# Async Usage

``UpdateTableManifestation`` supports async operations.



In [ ]:
import anyio

async def async_update_demo():
    """Demonstration of using the update table in asynchronous mode."""
    async_path = anyio.Path("tutorial_update_async.sqlite")
    if await async_path.exists():
        await async_path.unlink()

    async_db = LogDatabase(path=str(async_path), async_engine=True)
    await async_db.create_database_async()

    logs_async = async_db.logs

    # Async Insert
    await logs_async.insert_async({"event_type": "ASYNC_START", "payload": "Init", "update_id": 10})
    await logs_async.insert_async({"event_type": "ASYNC_END", "payload": "Done", "update_id": 11})

    # Async Last ID
    last_id = await logs_async.get_last_update_id_async()
    print(f"Async Last ID: {last_id}")

    # Async Delta Query
    # Note: get_from_update_async returns a Result object by default, which we can iterate over
    result = await logs_async.get_from_update_async(10, inclusive=False)

    print("Async logs after ID 10:")
    for item in result.scalars():
         print(f"[{item.update_id}] {item.event_type}")

    await async_db.close_async()
    await async_path.unlink()

# Run async demo
await async_update_demo()

In [ ]:
# Cleanup
database.close()
if db_path.exists():
    db_path.unlink()

# API Highlights

- **``BaseUpdateTableSchema``**: Adds ``update_id`` column.
- **``UpdateTableManifestation``**:
    - **``get_last_update_id()``**: Returns the max update_id.
    - **``get_from_update(update_id, inclusive=True)``**: Returns items with update_id >= (or >) the given ID.



# Troubleshooting / FAQs

- **Problem**: `get_from_update` returns empty.
  - **Solution**: Check if your ``update_id`` condition is correct. If ``inclusive=False``, it must be strictly greater. Also ensure you actually inserted data with ``update_id``s.



# Conclusion and Next Steps

You've learned how to use ``UpdateTable`` to efficiently track and query sequential updates.

- **Reference**: See ``docs/concepts/comprehensive.rst``.

